<a href="https://colab.research.google.com/github/YBenjaminPCondori/Machine-Learning-Gen-AI/blob/main/Occupancy-Monitoring-Conv1D-System-with-Gaussian-Data-Compression/Conv1D_Occupancy_Detection_Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Conv1D-Based Occupancy Detection from Multisensor Time-Series Data

## Abstract
This notebook demonstrates a Deep Learning approach to occupancy detection using **1D Convolutional Neural Networks (Conv1D)**. Unlike traditional Machine Learning models (e.g., Random Forest) that treat data points independently, Conv1D models exploit **temporal dependencies** in time-series data, making them ideal for sensor streams.

### Key Concepts Covered:
1.  **Temporal Feature Engineering**: Sliding window techniques.
2.  **Deep Learning for Time-Series**: Using Conv1D layers for feature extraction.
3.  **Handling Imbalanced Data**: Class weighting strategies.
4.  **Regularization**: Dropout and Batch Normalization.
5.  **Evaluation**: Precision, Recall, F1-Score in binary classification.

## 1. Configuration & Hyperparameters
Centralize configuration to make experiments reproducible and easy to tune.

In [ ]:
# --- File Paths ---
DATASET_PATH = r"C:/Users/ybenj/Downloads/ALL_CSVS/merged_env_motion.csv"

# --- Model Hyperparameters ---
TIME_STEPS = 100         # Look-back window size
BATCH_SIZE = 32          # Batch size for training
EPOCHS = 10              # Maximum training epochs
LEARNING_RATE = 0.001    # Adam Optimizer Learning Rate


## 2. Imports and Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.utils import class_weight
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, log_loss, roc_curve
)

import matplotlib.pyplot as plt
import seaborn as sns

# Set seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)


## 3. Data Preprocessing Functions

Separating data logic into functions allows for easier testing and cleaner execution flows.

In [ ]:
def load_and_clean_data(filepath):
    """
    Loads CSV, removes duplicates, handles missing values, and drops irrelevant columns.
    """
    print("Loading dataset...")
    df = pd.read_csv(filepath)
    
    # Cleaning
    df.drop_duplicates(inplace=True)
    df.columns = df.columns.str.strip().str.lower()
    df = df[df['motion'].notna()].copy()
    
    # Imputation
    numeric_cols = df.select_dtypes(include=np.number).columns
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
    
    # Feature Selection
    drop_cols = ['smoke', 'device', 'hub', 'home']
    if 'co' in df.columns: drop_cols.append('co')
    df.drop(columns=drop_cols, errors='ignore', inplace=True)
    
    # Type conversions
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp']).astype('int64') / 1e9
    if 'light' in df.columns and df['light'].dtype == bool:
        df['light'] = df['light'].astype(int)
        
    return df

def normalize_data(df, target_col='motion'):
    """
    Separates features/target and scales features using StandardScaler.
    """
    df[target_col] = df[target_col].astype(int)
    features = [col for col in df.columns if col != target_col]
    
    scaler = StandardScaler()
    df[features] = scaler.fit_transform(df[features])
    
    X = df[features].values.astype(np.float32)
    y = df[target_col].values.astype(np.float32)
    
    return X, y, len(features)


### Execute Data Loading

In [ ]:
# Execute the pipeline
df_clean = load_and_clean_data(DATASET_PATH)
X, y, num_features = normalize_data(df_clean)

print(f"Data Loaded. Shape: {X.shape}, Target Shape: {y.shape}")


## 4. Sliding Window Construction

**Theoretical Concept:** Time-series data is sequential. To predict the state at time $t$, we often need context from times $t-1, t-2, ..., t-k$.

In [ ]:
def create_windows(data, labels, window_size):
    """
    Converts flat time-series data into overlapping windows.
    """
    X_windows, y_labels = [], []
    for i in range(len(data) - window_size):
        X_windows.append(data[i:i+window_size])
        y_labels.append(labels[i+window_size])
    return np.array(X_windows), np.array(y_labels)

X_windowed, y_windowed = create_windows(X, y, TIME_STEPS)
print(f"Windowed Input Shape: {X_windowed.shape}")  


## 5. TensorFlow Dataset Pipeline

We use `tf.data` for efficient shuffling, batching, and prefetching.

In [ ]:
def create_tf_dataset(X, y, batch_size=32, split=0.8):
    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    dataset = dataset.shuffle(1000).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    total_batches = len(list(dataset))
    train_size = int(split * total_batches)

    train_ds = dataset.take(train_size)
    test_ds = dataset.skip(train_size)
    return train_ds, test_ds

train_ds, test_ds = create_tf_dataset(X_windowed, y_windowed, BATCH_SIZE)


## 6. Class Imbalance Handling

In [ ]:
def get_class_weights(y):
    class_weights_array = class_weight.compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y.astype(int)),
        y=y.astype(int)
    )
    return dict(enumerate(class_weights_array))

class_weights = get_class_weights(y_windowed)
print("Class Weights:", class_weights)


## 7. Conv1D Model Architecture

We encapsulate the model build process in a function to allow for easy hyperparameter tuning later.

In [ ]:
def build_conv1d_model(input_shape, activation='relu'):
    model = tf.keras.Sequential([
        # --- Block 1 ---
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(64, 3, activation=activation, padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling1D(2),

        # --- Block 2 ---
        tf.keras.layers.Conv1D(128, 3, activation=activation, padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling1D(2),

        # --- Block 3 ---
        tf.keras.layers.Conv1D(256, 3, activation=activation, padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling1D(2),

        # --- Classification Head ---
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation=activation),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(64, activation=activation),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_conv1d_model(input_shape=(TIME_STEPS, num_features))
model.summary()


## 8. Training

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=test_ds,
    callbacks=[early_stop],
    class_weight=class_weights
)


## 9. Evaluation & Visualization

In [ ]:
def plot_confusion_matrix(y_true, y_pred_classes):
    cm = confusion_matrix(y_true, y_pred_classes)
    
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
                xticklabels=['Empty', 'Occupied'], 
                yticklabels=['Empty', 'Occupied'])
    plt.title("Confusion Matrix")
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

def evaluate_model(model, test_ds):
    loss, accuracy = model.evaluate(test_ds)
    print(f"\nTest Loss: {loss:.4f}")
    print(f"Test Accuracy: {accuracy:.4f}")
    
    y_true, y_pred_probs, y_pred_classes = [], [], []
    for x_batch, y_batch in test_ds:
        preds = model.predict(x_batch, verbose=0).flatten()
        y_true.extend(y_batch.numpy())
        y_pred_probs.extend(preds)
        y_pred_classes.extend((preds > 0.5).astype(int))
        
    y_true = np.array(y_true)
    y_pred_classes = np.array(y_pred_classes)
    
    plot_confusion_matrix(y_true, y_pred_classes)
    print("\nClassification Report:\n")
    print(classification_report(y_true, y_pred_classes))

evaluate_model(model, test_ds)
